# YOLO 객체 탐지 실습
[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimhaneul050126-cmyk/ai-vision-lab/blob/main/02_YOLO_객체탐지_실습.ipynb)
MNIST에서는 사진 전체에 대해 숫자 하나를 예측했습니다. 여기서는 사진 안의 물체마다 **종류, 위치, 탐지 점수**를 확인합니다.

위에서부터 셀을 차례로 실행하세요. **기본 CPU 런타임**으로 충분합니다(GPU 변경 불필요). 첫 실행에는 패키지 설치와 모델 다운로드로 1~2분 걸립니다.

In [ ]:
# 수업용으로 동작을 확인한 버전으로 고정합니다
%pip -q install ultralytics==8.4.207

## 1. 모델 불러오기
공개 이미지 데이터(COCO, 80가지 물체)로 미리 학습된 **YOLO26n**을 사용합니다. 예제 사진은 패키지 안에 들어 있어 인터넷 연결 상태와 상관없이 열립니다.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps
from ultralytics import YOLO
from ultralytics.utils import ASSETS

model = YOLO("yolo26n.pt")
bus_photo = str(ASSETS / "bus.jpg")        # 버스와 사람들
people_photo = str(ASSETS / "zidane.jpg")  # 두 사람

def detect(photo, threshold=0.25):
    """사진 한 장을 탐지하고, 점수가 threshold 이상인 박스만 남긴 결과를 돌려줍니다."""
    return model.predict(source=photo, conf=threshold, device="cpu", verbose=False)[0]

def show(result):
    """박스가 그려진 사진을 화면에 표시합니다."""
    display(result.plot(pil=True))

def to_table(result):
    """탐지 결과를 표(종류·점수·좌표)로 바꿉니다."""
    rows = []
    for box in result.boxes.cpu():
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        rows.append({
            "종류": result.names[int(box.cls.item())],
            "탐지 점수": round(box.conf.item(), 3),
            "왼쪽 x": round(x1), "위 y": round(y1),
            "오른쪽 x": round(x2), "아래 y": round(y2),
        })
    columns = ["종류", "탐지 점수", "왼쪽 x", "위 y", "오른쪽 x", "아래 y"]
    return pd.DataFrame(rows, columns=columns)

print("모델이 아는 물체 수:", len(model.names))

## 2. 예제 사진에서 박스와 점수 읽기
박스는 모델이 예측한 위치입니다. 박스 위의 글자는 물체 종류이고, 숫자는 **탐지 점수**입니다.

- 탐지 점수는 "정답일 확률"이 아니라 **모델이 얼마나 확신하는지**를 나타내는 값입니다.
- 몇 점 이상을 결과로 인정할지(**기준, conf**)는 우리가 정합니다.
- 표의 좌표는 이미지 왼쪽 위를 (0, 0)으로 한 픽셀 위치입니다.

In [ ]:
result = detect(bus_photo)
show(result)
display(to_table(result))

## 3. 같은 사진, 다른 점수 기준
기준을 바꾸면 남는 박스가 달라집니다.

- 기준을 **낮추면**: 놓친 물체를 더 찾지만, 엉뚱한 박스(**오검출**)도 늘어납니다.
- 기준을 **높이면**: 엉뚱한 박스는 줄지만, 실제 물체를 놓치는 경우(**미검출**)가 늘어납니다.

세 그림을 비교해 **어떤 박스가 언제 사라지는지** 물체 이름과 점수를 함께 기록해 보세요.

In [ ]:
thresholds = (0.10, 0.25, 0.70)

fig, axes = plt.subplots(1, len(thresholds), figsize=(15, 7))
for ax, threshold in zip(axes, thresholds):
    prediction = detect(bus_photo, threshold)
    ax.imshow(prediction.plot(pil=True))
    ax.set_title(f"conf = {threshold:.2f} | {len(prediction.boxes)} boxes")
    ax.axis("off")
plt.tight_layout()
plt.show()

# 가장 낮은 기준에서 잡힌 박스 전체를 점수 순으로 확인
display(to_table(detect(bus_photo, thresholds[0])).sort_values("탐지 점수", ascending=False))

## 4. 다른 사진과 비교하기
같은 기준으로 두 번째 사진을 실행합니다. 사진 속 물체의 크기나 가려진 정도가 달라지면 탐지 결과도 달라질 수 있습니다.

In [ ]:
second_result = detect(people_photo, 0.25)
show(second_result)
display(to_table(second_result))

## 5. 탐지 결과를 RC카 명령으로 바꾸기
탐지 결과만으로는 차가 움직이지 않습니다. 결과를 보고 **명령(w 전진 / x 정지)**을 고르는 판단 코드가 필요합니다.

아래 함수는 사람이나 정지 표지판이 기준 점수 이상으로 보이면 정지(`x`), 아니면 전진(`w`)을 고릅니다. `stop_classes`와 `threshold`를 바꿔 보며 명령이 어떻게 달라지는지 확인하세요.

In [ ]:
def decide_from_detection(result, stop_classes=("person", "stop sign"), threshold=0.5):
    """멈춰야 할 물체가 기준 점수 이상으로 보이면 'x'(정지), 아니면 'w'(전진)."""
    for box in result.boxes:
        name = result.names[int(box.cls.item())]
        score = box.conf.item()
        if name in stop_classes and score >= threshold:
            print(f"정지 이유: {name} (점수 {score:.2f})")
            return "x"
    return "w"

# 낮은 기준(0.10)으로 탐지한 결과를 판단 함수에 넘깁니다
low_result = detect(bus_photo, 0.10)
print("명령:", decide_from_detection(low_result))
print("명령 (정지 표지판만, 기준 0.1):", decide_from_detection(low_result, stop_classes=("stop sign",), threshold=0.1))
print("명령 (정지 표지판만, 기준 0.5):", decide_from_detection(low_result, stop_classes=("stop sign",), threshold=0.5))

## 6. 내 사진으로 실행하기 (선택)
JPG 또는 PNG 한 장을 올립니다. 모델이 놓친 물체가 있다면 `threshold`를 낮추고, 어떤 박스가 새로 표시되는지 비교해 보세요.

In [ ]:
from google.colab import files

threshold = 0.25
uploaded = files.upload()
if len(uploaded) == 1:
    photo_bytes = next(iter(uploaded.values()))
    with Image.open(BytesIO(photo_bytes)) as image:
        photo = ImageOps.exif_transpose(image).convert("RGB")
    custom_result = detect(photo, threshold)
    show(custom_result)
    display(to_table(custom_result))
    print("명령:", decide_from_detection(custom_result))
else:
    print("이미지 한 장을 선택해 셀을 다시 실행하세요.")

## 결과 정리와 RC카 연결
- 기준을 올리면 어떤 박스가 사라졌는지 **물체 이름과 점수**를 함께 기록합니다.
- **미검출**: 실제로 있는 물체를 찾지 못한 경우입니다. **오검출**: 없는 물체를 있다고 한 경우입니다. 기준값은 둘 사이의 균형을 고르는 일입니다.
- 이 모델은 신호등(`traffic light`)의 **위치**는 찾지만 빨간불·초록불은 구분하지 않습니다. 색 판별은 별도로 하고, 그 결과를 **전진·정지 판단 코드**에 넘깁니다.
- 안전이 중요한 RC카에서는 "확신이 낮으면 멈춘다"가 기본입니다. 정지 판단의 기준은 낮게, 전진 판단의 기준은 높게 두는 이유를 생각해 보세요.

참고 자료 · [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26) · [Ultralytics Predict 모드](https://docs.ultralytics.com/modes/predict)

## 확장 실습 · 기준 점수와 안전 판단
사진 한두 장으로는 판단 함수가 **얼마나 자주 틀리는지** 알 수 없습니다. 정답 라벨이 있는 사진 **128장(COCO128)**에 판단 함수를 돌려서, 기준 점수를 바꿀 때 오류가 어떻게 변하는지 직접 셉니다.

- **멈춰야 하는 사진**: 정답 라벨에 사람(person) 또는 정지 표지판(stop sign)이 있는 사진
- **AI가 멈춤**: 그 물체를 기준 점수 이상으로 하나라도 찾은 경우

| | AI가 멈춤 (x) | AI가 지나감 (w) |
|---|---|---|
| **멈춰야 함** | 맞게 멈춤 (TP) | **못 멈춤 (FN)** |
| **지나가도 됨** | **괜히 멈춤 (FP)** | 맞게 지나감 (TN) |

### 단계 1 · 사진 128장에 탐지 한 번 돌리기
처음 한 번만 데이터(약 7MB)를 내려받습니다. 낮은 기준(0.05)으로 한 번 탐지해 두고, 사진마다 '멈출 물체'의 **가장 높은 점수**를 기록합니다. (CPU로 약 1분)

In [ ]:
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("coco128.yaml")
image_dir = Path(data["train"])
label_dir = image_dir.parent.parent / "labels" / image_dir.name
STOP_IDS = {0, 11}   # COCO 번호: 0 = person, 11 = stop sign

rows = []
for photo in sorted(image_dir.glob("*.jpg")):
    label_file = label_dir / (photo.stem + ".txt")
    lines = label_file.read_text().splitlines() if label_file.exists() else []
    must_stop = any(line.split() and int(line.split()[0]) in STOP_IDS for line in lines)
    result = detect(str(photo), 0.05)
    scores = [b.conf.item() for b in result.boxes if int(b.cls.item()) in STOP_IDS]
    rows.append({"사진": photo.name, "멈춰야 함": must_stop, "최고 점수": round(max(scores, default=0.0), 3)})

photos = pd.DataFrame(rows)
print(f"사진 {len(photos)}장 중 멈춰야 하는 사진 {photos['멈춰야 함'].sum()}장")
display(photos.head())

### 단계 2 · 기준 점수를 바꾸며 오류 세기
`기준 점수`를 넘으면 정지(x), 아니면 전진(w)으로 판단합니다. 표에서 **못 멈춤**과 **괜히 멈춤**이 어떻게 바뀌는지 보세요.

In [ ]:
def evaluate(threshold):
    stop = photos["최고 점수"] >= threshold
    truth = photos["멈춰야 함"]
    tp, fp = int((stop & truth).sum()), int((stop & ~truth).sum())
    fn, tn = int((~stop & truth).sum()), int((~stop & ~truth).sum())
    return {"기준 점수": threshold, "맞게 멈춤": tp, "괜히 멈춤": fp, "못 멈춤": fn, "맞게 지나감": tn,
            "정밀도": round(tp / (tp + fp), 3) if tp + fp else 0.0,
            "재현율": round(tp / (tp + fn), 3) if tp + fn else 0.0}

THRESHOLDS = [0.1, 0.25, 0.5, 0.7, 0.9]
summary = pd.DataFrame([evaluate(t) for t in THRESHOLDS])
display(summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary["기준 점수"], summary["못 멈춤"], "o-", color="#d62728", lw=2.5, label="못 멈춤")
ax.plot(summary["기준 점수"], summary["괜히 멈춤"], "o-", color="#1f4fd8", lw=2.5, label="괜히 멈춤")
ax.set_xlabel("기준 점수"); ax.set_ylabel("사진 수"); ax.set_xticks(THRESHOLDS)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e5e5e5")
ax.legend(frameon=False); plt.show()

### 단계 3 · 못 멈춘 사진 직접 보기
기준 0.5에서 **멈춰야 했는데 지나간** 사진입니다. 사람이 작거나, 가려졌거나, 잘려 있지 않은지 살펴보세요.

In [ ]:
CHECK = 0.5
missed = photos[photos["멈춰야 함"] & (photos["최고 점수"] < CHECK)]
print(f"기준 {CHECK}에서 못 멈춘 사진: {len(missed)}장")
fig, axes = plt.subplots(1, min(4, len(missed)) or 1, figsize=(16, 4))
for ax, (_, row) in zip(list(axes.ravel()) if hasattr(axes, "ravel") else [axes], missed.head(4).iterrows()):
    ax.imshow(Image.open(image_dir / row["사진"])); ax.axis("off")
    ax.set_title(f"{row['사진']} · 최고 점수 {row['최고 점수']}", fontsize=9)
plt.show()

### 단계 4 · 점수를 명령 글자(w / x)로 바꾸기
위에서 고른 기준 점수를 `MY_THRESHOLD`에 넣고 실행합니다. 앞에서 만든 판단 함수 `decide_from_detection`이 사진마다 명령 글자를 정합니다.

- 사람·정지 표지판 점수 **≥ 기준 점수** → `x` (정지)
- 그 밖의 경우 → `w` (전진)

COCO128에서 **도로 사진 4장**을 골라 RC카 카메라 화면이라고 생각하고 판단해 봅니다.

- **파란 상자**: 기준 점수를 넘어 정지 이유가 된 사람·정지 표지판
- **회색 점선 상자**: 찾긴 했지만 점수가 기준보다 낮은 것
- 아래 글자가 **빨간색**이면 판단이 정답과 다른 사진입니다.

In [ ]:
import matplotlib.patches as patches

MY_THRESHOLD = 0.5   # ← 우리 팀이 고른 기준 점수로 바꿔 보세요
ROAD_PHOTOS = ["000000000532.jpg", "000000000257.jpg", "000000000641.jpg", "000000000094.jpg"]

fig, axes = plt.subplots(1, len(ROAD_PHOTOS), figsize=(4.5 * len(ROAD_PHOTOS), 4))
for ax, name in zip(axes, ROAD_PHOTOS):
    result = detect(str(image_dir / name), 0.1)
    command = decide_from_detection(result, threshold=MY_THRESHOLD)
    truth = "x" if photos.set_index("사진").loc[name, "멈춰야 함"] else "w"
    ax.imshow(Image.open(image_dir / name)); ax.axis("off")
    for box in result.boxes:
        if int(box.cls.item()) not in STOP_IDS:
            continue
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        strong = box.conf.item() >= MY_THRESHOLD
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor="#1f4fd8" if strong else "#555555",
                                       linewidth=2.5 if strong else 2.0, linestyle="-" if strong else "--"))
    ax.set_title(f"명령 {command}  ·  정답 {truth}", y=-0.12,
                 color="black" if command == truth else "#d62728", fontsize=13)
plt.tight_layout(); plt.show()

counts = evaluate(MY_THRESHOLD)
print(f"128장 전체에서 기준 {MY_THRESHOLD}이면 → 못 멈춤 {counts['못 멈춤']}장, 괜히 멈춤 {counts['괜히 멈춤']}장")

### 단계 5 · 조종기의 안전 규칙
AI 판단과 별개로, 조종기 쪽에도 **애매하면 멈춘다**는 규칙이 있습니다. 아래 함수는 RC카 v2 조종기가 글자를 받았을 때 하는 일을 흉내 낸 것입니다.

- 처음 켜지면 정지 상태
- `w`만 전진, 약속하지 않은 글자는 정지
- 명령이 0.7초(700ms) 동안 오지 않으면 자동으로 정지

In [ ]:
COMMAND_HOLD_MS = 700

def car_action(command, ms_since_last_command=0):
    if ms_since_last_command > COMMAND_HOLD_MS:
        return "정지 (명령이 끊김)"
    if command == "w":
        return "전진"
    if command == "x":
        return "정지"
    return f"정지 (모르는 글자 {command!r})"

for command, wait in [("w", 0), ("x", 0), ("a", 0), ("w", 1000)]:
    print(f"받은 글자 {command!r}, {wait}ms 뒤 → {car_action(command, wait)}")

### 정리 · RC카에는 어떤 기준이 맞을까?
- **못 멈춤**은 사고로 이어질 수 있고, **괜히 멈춤**은 불편하지만 안전합니다.
- 그래서 정지 판단은 보통 **재현율(못 멈춤을 줄이기)**을 우선해 기준을 낮게 잡습니다. 대신 괜히 멈춤이 얼마나 늘어나는지도 함께 봅니다.
- 팀별로 기준 점수 하나를 고르고, 그 기준에서 **못 멈춤 수 · 괜히 멈춤 수 · 고른 이유**를 한 줄로 적어 보세요.
- 단계 4의 `w`/`x`를 시리얼로 조종기에 보내면 실제 RC카가 움직입니다. 그래도 놓치는 사진은 **우리 환경 사진으로 다시 학습**시켜 줄입니다. 2부에서 이어집니다.

데이터 · [COCO128](https://docs.ultralytics.com/datasets/detect/coco128/) (COCO 2017 train 앞 128장, CC BY 4.0)